In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260603_044712"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots_new.parquet"))

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1780397822614,BTCUSDT,69684.01,69684.02,69684.015,69684.014322,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
1,1780397822714,BTCUSDT,69684.01,69684.02,69684.015,69684.014322,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
2,1780397822814,BTCUSDT,69684.01,69684.02,69684.015,69684.014322,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
3,1780397822914,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
4,1780397823014,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15245,1780399347114,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,NaN,NaN,NaN,NaN,NaN,NaN
15246,1780399347214,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,NaN,NaN,NaN,NaN,NaN,NaN
15247,1780399347314,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,NaN,NaN,NaN,NaN,NaN,NaN
15248,1780399347414,BTCUSDT,69466.00,69466.01,69466.005,69466.008130,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
"""
model = mid + struct_delta + micro_signal + ml_delta (y here), mid is baseline for quoting center

Does ml_delta explain the residual error in my reservation price (mid + struct_delta + micro_signal)?

"""

df = snapshots

horizons = [100, 500, 1000, 5000]

for h in horizons:
    df[f"y_{h}ms"] = np.log(df[f"future_mid_{h}ms"] / df["reservation"]) # residuals with only reservation = mid + struct_delta + micro_drift

df["residual_mid"] = (df["reservation"] - df["mid"]) / df["mid"]
df["micro_residual"] = (df["microprice"] - df["reservation"]) / df["reservation"]
df["micro_signal"] = (df["microprice"] - df["mid"]) / df["mid"]

feature_cols = [
    # raw microstructure
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "inventory",
    "volatility",
    "queue_ahead_bid",
    "queue_ahead_ask",

    # very important: model error signals
    "residual_mid",
    "micro_residual",
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols]
X_test = test[feature_cols]

In [ ]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}

for h in horizons:

    y_train = train[f"y_{h}ms"]
    y_test = test[f"y_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

results_df = pd.DataFrame(results)
results_df

c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\admin\AppData\Local\Temp\ipykernel_35496\1703732673.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(pred, y).statistic
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\admin\AppData\Local\Temp\ipykernel_35496\170

,100,500,1000,5000
Residual_IC,NaN,NaN,NaN,1.987458e-01
Residual_RankIC,NaN,NaN,NaN,1.782180e-01
HitRate,5.019737e-01,5.220395e-01,5.424342e-01,5.398026e-01
PnLProxy,9.798839e-14,1.015003e-12,3.663439e-12,6.522907e-10
SharpeProxy,9.747447e-05,9.757888e-04,3.316581e-03,1.646511e-01


In [13]:
artifact = {
    "model": model,
    "feature_cols": feature_cols,
    "target": "log(future_mid/reservation)",
    "horizon_ms": 5000,
}

joblib.dump(artifact, "residual_ml_model.pkl")

['residual_ml_model.pkl']

In [ ]:
"""

You should align horizon to signal:

Recommended:
micro_signal → 100-500ms
struct_delta → 500-2000ms
ML residual → 1000-5000ms

You already discovered this implicitly in IC results.

mid
│ │ Inventory       │  1.8157                                       │ │
│ │ Realized PnL    │ ▼ -54.0738                                    │ │
│ │ Unrealized PnL  │ ▼ -251.6730                                   │ │
│ │ Total PnL       │ ▼ -305.7468                                   │ │

mid + micro_signal
│ │ Inventory       │    -2.5434                                    │ │
│ │ Realized PnL    │ ▲ 506.5967                                    │ │
│ │ Unrealized PnL  │ ▲ 494.6997                                    │ │
│ │ Total PnL       │ ▲ 1001.2965                                   │ │
sharpe -0.004150854169171361

mid + struct_delta
│ │ Inventory       │  -1.2852                                      │ │
│ │ Realized PnL    │ ▲ 1218.6432                                   │ │
│ │ Unrealized PnL  │ ▲ 270.5966                                    │ │
│ │ Total PnL       │ ▲ 1489.2398                                   │ │  

mid + struct_delta + micro_signal
│ │ Inventory       │  -0.9532                                      │ │
│ │ Realized PnL    │ ▲ 2756.1612                                   │ │
│ │ Unrealized PnL  │ ▲ 330.4319                                    │ │
│ │ Total PnL       │ ▲ 3086.5930                                   │ │
SHARPE: 0.015884977143609876

mid + struct_delta + micro_signal + ml_delta
│ │ Inventory       │  -1.1747                                      │ │
│ │ Realized PnL    │ ▲ 1614.8600                                   │ │
│ │ Unrealized PnL  │ ▲ 369.1110                                    │ │
│ │ Total PnL       │ ▲ 1983.9710                                   │ │
SHARPE: 0.023788313502426443


Why your Sharpe improved but PnL dropped

This is actually very important:

Signal stack	Behavior
mid only	unbiased but noisy
+ micro	directional edge but noisy inventory
+ struct	improves mean reversion capture
+ ML delta	reduces overtrading + improves timing

So what happened:
ML signal likely:
reduces trades in bad regimes
flattens exposure faster
improves timing

but:
removes some high-volatility alpha trades

So:
you improved risk-adjusted edge, not raw edge

That's exactly what ML residual models usually do first.
""" 

In [ ]:
# # Notes

# quotes are not getting filled at all with quoting logic -> not competitive
# make spread adaptive

# 1. Your fill model is currently unrealistic

# This is the most important weakness.

# Right now:

# if price == state.bid_quote:

# This assumes:

# every trade at your price may fill you
# but ignores queue position

# In real HFT:

# queue position is EVERYTHING
# most passive orders never fill
# adverse selection dominates

# Right now your code is evolving into a real event-driven MM simulator, but the architecture is still “research notebook style” rather than “exchange engine style.” For HFT/quant interviews, the separation of concerns matters almost as much as the alpha logic.

# The strongest version of this project is:

# Market Data Layer
# Strategy Layer
# Execution / Simulation Layer
# (optional but very strong) Risk Layer

# That architecture immediately signals:

# systems thinking
# low-latency awareness
# production-style design
# extensibility
# understanding of real trading stacks

# Your current code already contains these layers conceptually — they’re just intermingled.

# to generate dataset for ML fill model, queue position is estimate heuristically based on level size and a position factor (e.g., 0.3). This is a simplification, but it allows you to create a feature that captures the idea of “how much liquidity is ahead of me at this price level?” which is crucial for fill probability estimation.

# Need backtesting ML layer to generate fill probability

# Why this stops your infinite failure loop
# You were previously:
# restarting entire system → destroying continuity
# hoping for overlap event → statistically rare
# desyncing every attempt
# Now:
# single lifecycle
# deterministic timeout
# controlled sync window
# no recursive restart corruption
# Final insight (important for your interview)

# If you say this in an interview, it signals senior-level thinking:

# “My initial bug was treating L2 sync as a stateless matching problem, but it is actually a stateful streaming alignment problem requiring a single-lifecycle reconciliation window, not repeated restart attempts.”

# my quoting prices were only up to 2 decimal places, but the exchange operates at 0.01 tick size, so I was effectively quoting at 0.01 increments but with a lot of rounding noise, which made my quotes non-competitive and rarely filled. By implementing a proper tick conversion and ensuring all prices are aligned to the exchange’s tick size, I can now quote more accurately and increase my chances of getting filled.

# What a 1-tick spread actually implies

# At:

# Bid = 77799.99
# Ask = 77800.00

# This is:

# ultra-tight market
# extremely fast queue turnover
# heavy competition at top of book
# lots of passive liquidity stacking

# So:

# If you place:
# bid = 77799.99 → you join a massive queue
# ask = 77800.00 → you join another massive queue

# You are now competing with:

# market makers co-located on Binance infra
# algos reacting in milliseconds
# constant cancellations/replacements

# compute_queue_ahead with level * 0.3 is too static, i should model FIFO correctly, if not im always not getting filled.

# learnt that on depth is to update order book and queue position, while on trade is to check if i got filled and update inventory/cash. This separation is crucial for accurate state management and realistic simulation of market dynamics.

# place_quotes should not be called on depth as it might be over requoting, it should run independently on a timer or certain conditions to avoid overfitting to every market update and to simulate more realistic trading behavior.

# adverse selection is around 75% of fills

# will need to log in attribution everytime on_fill

# also might need a regime detection

# Got worse PNL when rounding then adjusting when generate_quotes, rather than adjusting then adjusting
# “Because in a discrete limit order book, rounding is not a cosmetic step — 
# it defines execution price priority and queue position. 
# Changing when discretization happens alters spread formation and fill sequencing, which dominates PnL more than the alpha signal itself.”

# realised execution latency matters, did not include this

# react dashboard, as rich terminal jittery as more and more metric put out, use react to run own metrics

# realized i need to reliably run the same datasets, with the same config for backtesting smoothly -> need for modular architecture that can ingest different configs -> use of manifest file

# currently execution step is driven by polling, not on market data. will change so it will not react to stale quotes, moved on market data to ASYNC

# during dataset testing, signal to check if microprice is a good indicator of mid is too noisy, corr was too small and dataset was too small. moved back to primitive mid prediction

# model is currently using fair and skew to compute microprice. 

# since microprice is a good fair value estimator of future mid, can we estimate the adjusted microprice better with a custom alpha signal model?

# we can also detect regimes using a classifier to tweak inputs for our fair, skew, spread, struct delta, ml delta signals

# implemented signal quality to adaptively scale ml delta time horizon signal

# utilised mispricing in mid to generate micro_signal delta, using struct signal delta and ml signal delta for any residual signals to price reservation